# 8. Conclusions and Future Work

## 8.1 Summary of contributions

This chapter applied deep reinforcement learning to the topology-optimisation problem formalised in Chapter 4. The contributions are:

1. **SeqTO-v2 environment** (Section 6) — a 3-channel `[material, flux, boundary]` tensor extension of Chapter 4's `CcoreSeqTOEnv`, with a Maxwell-stress force reward. Preserves Chapter 4's chromosome-as-action-sequence formulation, connectivity-by-construction guarantee, and `MAX_STEPS = 34` budget.
2. **Four DRL agent implementations** (Sections 4–5) — DQN (`dqn_seqto.py`), A2C (`a2c_seqto.py` + `a2c_v2.py`), and PPO (`ppo_seqto.py` + `ppo_v2.py`) sharing the CNN backbone in `net_seqto.py`. The `_v2` scripts add a single `--reward-scale` knob to isolate value-loss explosion as a failure mode independent of algorithm choice.
3. **Environment diagnostics** (Section 6.8) — four probes (`determinism`, `random-search`, `action-effect`, `random-policy`) that quantify what the policy is up against *before* training begins. The `action-effect` probe (single-step rewards are nearly action-blind) turned out to be the most informative — it predicted the algorithm ordering observed in the training runs.
4. **A completed DQN training run** (Section 4.7) — 200 episodes, peak end-of-episode force 1305 N, ~159 h FEMM wall-clock. This is the headline result of the chapter.
5. **Reward-scaling intervention** (Section 5.9) — diagnosed the value-loss explosion + entropy-collapse failure mode that derails unscaled A2C / PPO on this env, and showed that a 0.001 multiplier on rewards restores stable training without any other change to the algorithm or hyperparameters.
6. **Cross-method comparison** (Section 7) — running-best force overlay, scorecard, mask gallery, hyperparameter table, and wall-clock cost breakdown across all seven RL runs plus the random-search baseline.

## 8.2 Key findings

**DQN solves this problem; the policy methods don't quite.** DQN reaches 1305 N peak end-of-episode force ($2.14\times$ the 610 N random-search ceiling), with the late-stage mean stabilising around 700 N. PPO-v2 reaches $\sim 795$ N and A2C-v2 plateaus near $\sim 764$ N — both beat random search, but DQN beats them by 60-70 %.

**The ordering follows from the env's action-effect structure, not from algorithm sophistication.** The `action-effect` probe (Section 6.8) showed that single-step rewards do not distinguish the 4 controller actions — all four produce overlapping force distributions with means within 3 N of each other. The useful signal lives in *multi-step compositions*, which means an algorithm's ability to extract that signal scales with how many gradient updates it makes per FEMM solve:

| Algorithm | Gradient updates per FEMM solve | Final-run peak (N) |
|---|---:|---:|
| DQN (replay buffer revisits each transition) | $\sim 30$ | **1305** |
| PPO (K=4 epochs on each rollout) | $\sim 4/34$ | 795 |
| A2C (one MC update per episode) | $\sim 1/34$ | 764 |

**Reward scaling is necessary but not sufficient for stable policy-gradient training on this env.** Unscaled A2C and PPO both collapse within $\sim 50$ episodes (entropy $\to 10^{-20}$, value-loss $\to 10^7$). Scaled `_v2` variants train productively — but they still don't catch up to DQN. The scaling fix is a *prerequisite* for the policy methods to be usable at all; it is not what closes the gap to DQN.

**SB3 reference implementations were not faster, cheaper, or more correct than the custom scripts.** SB3-PPO undertrained at the default 10k-step budget (peak 339 N — below the random-search ceiling), and SB3-A2C with `reward_scale=1.0` got stuck at its second episode (267 N peak), confirming the same value-loss failure mode that derailed our own raw-reward A2C. This is consistent with the literature finding that SB3's default hyperparameters are tuned for Atari / MuJoCo scales, not for $O(10^4)$-magnitude environment-specific rewards. A fair SB3 comparison requires reward scaling at the env level.

## 8.3 Connection to Chapter 4

The action-effect finding (Section 6.8) is the deep-RL generalisation of the credit-assignment problem Chapter 4 ran into with tabular Q-learning on SeqTO-v1. Both problems share the same structural difficulty:

- **Per-step reward signal is weak.** In Chapter 4's SynRM env, the per-step torque change was small and noisy relative to inter-mask variance; in Chapter 5's C-core env, the per-step force change is essentially zero — the four actions are statistically indistinguishable after one step.
- **Multi-step composition is what matters.** In both chapters, the productive policies were the ones that composed long sequences of actions into globally good masks, not the ones that made locally good single-step decisions.
- **Algorithms that re-use signal beat algorithms that don't.** Chapter 4's `q_learn_improved.py` succeeded against the tabular baseline largely because of its $\varepsilon$-decay-then-replay-the-good-trajectories mechanism; Chapter 5's DQN succeeds against A2C and PPO largely because of its replay buffer. The same idea — extract more gradient signal per FEMM solve — works at both the tabular and deep-network scales.

So Chapter 5 isn't a new method on a new problem; it's the same problem with a richer state representation and a CNN function approximator. The chapter-level lesson is that **the env's credit-assignment difficulty constrains algorithm choice far more than the function approximator does**.

## 8.4 Planned future work

In priority order. All items are tracked rather than executed in this jupyter-book session.

1. **DQN evaluation pass.** Load `designs/dqn_run/checkpoint.pt`, fix the policy (greedy argmax), and roll out 20-50 evaluation episodes from held-out random starts. The mean force from that pass is the number that should populate any benchmark — the 1305 N peak is an upper bound the agent has demonstrated reachable, not a steady-state operating point. ~1 h on the target box.
2. **PPO-v2 extended run.** PPO-v2 had only $\sim 150$ episodes when its entropy started to fall in the late tail. A 300-500 episode rerun with `--entropy-coef-final 0.02 --clip-eps 0.1` would establish whether PPO-v2 can match DQN with more compute, or whether the gap is structural.
3. **Random-policy and cell-flips probes.** The `random-policy` subcommand crashed at episode 13/50; the `cell-flips --from-best designs/dqn_run/best_mask.npy` probe was not run. Both are needed for a complete env-characterisation chapter.
4. **SB3 reruns with reward scaling.** The current SB3 baselines are unfair to SB3 — they were run with the library default `reward_scale=1.0`, which produces the same value-loss failure mode that derailed our own raw-reward runs. Rerunning SB3-A2C and SB3-PPO with `--reward-scale 0.001` is a $\sim 24$ h fix.
5. **Newton-calibrated cross-chapter comparison.** Re-evaluate the Chapter 4 GA-best and Q-learn-best masks under the SeqTO-v2 `reward_kind='force'` setting (same env, same Maxwell-stress integral) so the Chp4-vs-Chp5 comparison stops mixing units. This is the same Maxwell-stress recompute deferred in Chapter 4 §7.6.3.
6. **DRL on the SynRM polar domain.** Chapter 4 §6.10 has the SynRM polar env (`seqTO_SynRM_env.py`) with a `--grid-size N` flag; porting `dqn_seqto.py` and `ppo_v2.py` to that env would extend the DRL-vs-classical comparison to a second machine.
7. **Hyperparameter sweeps.** Beyond the reward-scale fix: learning-rate $\alpha$ for the policy methods (currently $3 \!\times\! 10^{-4}$ for DQN/PPO, $10^{-3}$ for A2C-v2 after one mid-run bump), DQN target-network $\tau$ (currently 0.005), and DQN $\gamma$. Deferred because the chapter's headline result (DQN, 1305 N) does not yet warrant the compute spend.

## 8.5 Limitations of this chapter

- **No evaluation passes.** Every number in Section 7 is a *training-time* best, not a steady-state evaluation. The peak force is an upper bound on reachable performance, not a guaranteed operating point. Item 1 above addresses this.
- **Single seed per run.** No seed-sensitivity analysis. The 1305 N DQN result might be reproducible across seeds or might be lucky; the data on disk cannot distinguish those.
- **C-core only.** SeqTO-v2 is C-core-only as written; the SynRM polar domain from Chapter 4 §6.10 is not addressed (item 6 above).
- **Unfair SB3 baseline.** As noted in §8.2, the SB3 runs used `reward_scale=1.0`, which is the same setting that derailed our own raw-reward A2C / PPO. The chapter's "SB3 underperforms our custom code" framing is therefore weaker than it looks — what's really being said is "SB3 with default config underperforms". Item 4 above addresses this.
- **Cross-chapter unit mismatch.** Chapter 4 reports mean-$|B|$-based reward; Chapter 5 reports force-in-N. The cross-chapter row in Section 7 is unit-inconsistent until item 5 above is done.

## 8.6 Broader takeaway

Two things this chapter argues, both grounded in the data on disk rather than in algorithm-on-paper analysis:

1. **Env characterisation pays for itself.** The four env-probes from Section 6.8 took $\sim 7$ FEMM-hours combined; their result (single-step rewards are action-blind) predicted the entire chapter ordering of algorithms ($DQN > PPO > A2C$, in step with how aggressively each method reuses per-step signal). Running the probes before the algorithm experiments would have set realistic expectations for A2C and accelerated the diagnosis of the unscaled-reward failure mode.
2. **The hard part is matching the algorithm to the env's structure, not picking a sophisticated algorithm.** PPO is the more sophisticated algorithm; DQN won. The reason isn't about clipped surrogates or trust regions — it's about which mechanism extracts the most gradient signal per expensive simulator call. On an env where FEMM solves cost $\sim 84$ s each, that's the metric that matters.


## References

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
- Mnih, V., et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518.
- Mnih, V., et al. (2016). Asynchronous methods for deep reinforcement learning. *ICML*.
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal Policy Optimization Algorithms. *arXiv:1707.06347*.
- Schulman, J., Moritz, P., Levine, S., Jordan, M., & Abbeel, P. (2016). High-Dimensional Continuous Control Using Generalized Advantage Estimation. *ICLR*.
- Raffin, A., Hill, A., Ernestus, M., Gleave, A., Kanervisto, A., & Dormann, N. (2021). Stable-Baselines3: Reliable Reinforcement Learning Implementations. *Journal of Machine Learning Research*.
- Sigmund, O., & Maute, K. (2013). Topology optimization approaches. *Structural and Multidisciplinary Optimization*, 48(6).
